# FinBERT Fine-Tuning on MD&A Sentiment (MPS GPU)

**Pipeline**

1. Load the zero-shot labelled dataset (`mda_processed_sample_zeroshot_labeled.xlsx`) — same source as `naivebayes.ipynb`
2. Reshape wide → long (one row = one sentence + ground-truth label)
3. Configure FinBERT architecture (layers / attention heads / hidden size)
4. Fine-tune `ProsusAI/finbert` on MPS (Apple Silicon GPU) with minimal hyperparameters
5. Evaluate on held-out test set


## 1. Imports & Device Setup


In [1]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset as HFDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device: MPS (Apple Silicon) → CUDA → CPU ─────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

PyTorch : 2.10.0
Device  : mps


## 2. Load & Prepare Dataset 

In [4]:
import ast
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

CSV_PATH = "datasets/cleaned/mda_shared_preprocessed.csv"

docs = pd.read_csv(CSV_PATH)
docs.head()

,doc_id,company_name,filing_type,filing_date,year,quarter,sentences,clean_text,n_sentences
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,"[""executive summary petmed express was incorpo...",executive summary petmed express was incorpora...,125
1,1-800-PetMeds_10-K_2020-05-26,1-800-PetMeds,10-K,2020-05-26,2020,Q2,"[""the companys common stock is traded on the n...",the companys common stock is traded on the nas...,149
2,1-800-PetMeds_10-Q_2020-08-03,1-800-PetMeds,10-Q,2020-08-03,2020,Q3,"[""executive summary petmed express was incorpo...",executive summary petmed express was incorpora...,103
3,1-800-PetMeds_10-Q_2020-11-03,1-800-PetMeds,10-Q,2020-11-03,2020,Q4,"[""executive summary petmed express was incorpo...",executive summary petmed express was incorpora...,124
4,1-800-PetMeds_10-Q_2021-01-26,1-800-PetMeds,10-Q,2021-01-26,2021,Q1,"[""executive summary petmed express was incorpo...",executive summary petmed express was incorpora...,123


In [5]:
print(f"Loaded : {docs.shape[0]:,} documents, {docs.shape[1]} columns")
print(f"Columns: {docs.columns.tolist()}")
print(f"Companies : {docs['company_name'].nunique():,}")
print(f"Year range: {docs['year'].min()} – {docs['year'].max()}")
print()
print("Filing type counts:")
print(docs["filing_type"].value_counts().to_string())



Loaded : 18,183 documents, 9 columns
Columns: ['doc_id', 'company_name', 'filing_type', 'filing_date', 'year', 'quarter', 'sentences', 'clean_text', 'n_sentences']
Companies : 473
Year range: 2010 – 2025

Filing type counts:
filing_type
10-Q    13723
10-K     4460


In [3]:
docs.head()

,doc_id,company_name,filing_type,filing_date,year,quarter,sentences,clean_text,n_sentences
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,"[""executive summary petmed express was incorpo...",executive summary petmed express was incorpora...,125
1,1-800-PetMeds_10-K_2020-05-26,1-800-PetMeds,10-K,2020-05-26,2020,Q2,"[""the companys common stock is traded on the n...",the companys common stock is traded on the nas...,149
2,1-800-PetMeds_10-Q_2020-08-03,1-800-PetMeds,10-Q,2020-08-03,2020,Q3,"[""executive summary petmed express was incorpo...",executive summary petmed express was incorpora...,103
3,1-800-PetMeds_10-Q_2020-11-03,1-800-PetMeds,10-Q,2020-11-03,2020,Q4,"[""executive summary petmed express was incorpo...",executive summary petmed express was incorpora...,124
4,1-800-PetMeds_10-Q_2021-01-26,1-800-PetMeds,10-Q,2021-01-26,2021,Q1,"[""executive summary petmed express was incorpo...",executive summary petmed express was incorpora...,123


In [ ]:
import json

# ── Parse sentence lists (json.loads is ~5× faster than ast.literal_eval) ─────
def _parse_list(val):
    try:
        return json.loads(val)
    except (json.JSONDecodeError, TypeError):
        import ast
        return ast.literal_eval(val)

docs["sentences"] = docs["sentences"].apply(_parse_list)

long_df = (
    docs.explode("sentences")
        .rename(columns={"sentences": "text"})
        .reset_index(drop=True)
)

long_df["text"] = long_df["text"].astype(str).str.strip()
long_df = long_df[long_df["text"].str.len() > 0].reset_index(drop=True)

# ── Optional: cap sentences to avoid OOM / runaway training time ──────────────
# Set MAX_SENTENCES = None to use all data.
MAX_SENTENCES = 500_000
if MAX_SENTENCES and len(long_df) > MAX_SENTENCES:
    long_df = (
        long_df.groupby("year", group_keys=False)
               .apply(lambda g: g.sample(frac=MAX_SENTENCES / len(long_df), random_state=SEED))
               .reset_index(drop=True)
    )
    print(f"Sampled down to {len(long_df):,} sentences (stratified by year)")

print(f"Documents : {docs.shape[0]:,}")
print(f"Sentences : {len(long_df):,}  (one row each)")
long_df[["doc_id", "company_name", "filing_type", "year", "quarter", "text"]].head(5)


## 3. FinBERT Architecture Configuration

FinBERT is built on **BERT-base** (`BertConfig`). Below are the four structural hyperparameters you can define — everything else is fixed by the pre-trained checkpoint.

| Parameter             | Default (FinBERT) | What it controls                                                                 |
| --------------------- | ----------------- | -------------------------------------------------------------------------------- |
| `num_hidden_layers`   | 12                | Number of Transformer encoder blocks stacked on top of each other                |
| `num_attention_heads` | 12                | Parallel attention "heads" per layer — each learns different token relationships |
| `hidden_size`         | 768               | Width of every hidden vector — **must be divisible by `num_attention_heads`**    |
| `intermediate_size`   | 3072              | Inner width of the feed-forward sub-layer (conventionally 4 × `hidden_size`)     |

**Total parameter count** scales roughly as:
`params ≈ num_layers × (4 × hidden² + 2 × hidden × intermediate)`

> **Important**: If you change `num_hidden_layers` / `hidden_size` / `num_attention_heads` away from the defaults, the pre-trained weights **cannot be loaded** for those layers — training starts from random. The values below match the original FinBERT so pre-trained weights are fully reused.


In [4]:
MODEL_NAME = "ProsusAI/finbert"

# ── Architecture knobs ────────────────────────────────────────────────────────
# Change these to reshape the model; keep defaults to reuse all pre-trained weights.
NUM_HIDDEN_LAYERS = 12  # Transformer blocks  (reduce → faster, less expressive)
NUM_ATTENTION_HEADS = 12  # Heads per block     (hidden_size must be divisible by this)
HIDDEN_SIZE = 768  # Hidden vector width (reduce together with intermediate_size)
INTERMEDIATE_SIZE = 3072  # FFN inner dim       (keep at 4 × hidden_size)

# ── Label mapping ─────────────────────────────────────────────────────────────
LABEL2ID = {"positive": 0, "neutral": 1, "negative": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# ── Tokenisation ──────────────────────────────────────────────────────────────
MAX_LEN = 128  # FinBERT paper uses 128; longer = quadratic attention cost

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── Load pre-trained FinBERT, overriding architecture params ──────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    # The four structural params — edit NUM_* constants above to customise
    num_hidden_layers=NUM_HIDDEN_LAYERS,
    num_attention_heads=NUM_ATTENTION_HEADS,
    hidden_size=HIDDEN_SIZE,
    intermediate_size=INTERMEDIATE_SIZE,
    ignore_mismatched_sizes=True,  # required when architecture differs from checkpoint
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(
    f"Architecture : {NUM_HIDDEN_LAYERS} layers × {NUM_ATTENTION_HEADS} heads × {HIDDEN_SIZE}d hidden"
)
print(f"Total params : {total_params:,}")
print(f"Trainable    : {trainable:,}")
print(f"Device       : {next(model.parameters()).device}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Architecture : 12 layers × 12 heads × 768d hidden
Total params : 109,484,547
Trainable    : 109,484,547
Device       : mps:0


## 4. Generate Zero-Shot Pseudo-Labels

Since no hand-labeled dataset is available, we run the **pre-trained** FinBERT in inference mode over all sentences to generate pseudo-labels.  
Fine-tuning then adapts the model to the domain-specific language of MD&A filings, going beyond what the general-purpose checkpoint learned.


In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE_INFER = 128  # larger batch → better GPU/MPS utilisation

# Use half precision for faster inference on CUDA; MPS stays at float32
infer_dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
model = model.to(dtype=infer_dtype)
model.eval()

all_texts = long_df["text"].tolist()
pseudo_labels = []

with torch.inference_mode():  # slightly faster than no_grad; disables autograd completely
    for i in tqdm(range(0, len(all_texts), BATCH_SIZE_INFER), desc="Zero-shot inference"):
        batch = all_texts[i : i + BATCH_SIZE_INFER]
        enc = tokenizer(
            batch,
            truncation=True,
            max_length=MAX_LEN,
            padding=True,
            return_tensors="pt",
        ).to(DEVICE)
        logits = model(**enc).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        pseudo_labels.extend([ID2LABEL[int(p)] for p in preds])

# Restore float32 for training (Trainer handles its own mixed precision)
model = model.to(dtype=torch.float32)

long_df["label"] = pseudo_labels
long_df["label_id"] = long_df["label"].map(LABEL2ID)

print("Zero-shot label distribution:")
print(long_df["label"].value_counts())


### 4a. Label Proportion EDA

Four subplots showing how sentiment is distributed:

- **Overall** proportion
- **By filing type** (10-K vs 10-Q)
- **By year**
- **By quarter** (Q1–Q4)


## 5. Time-Stratified Train / Test Split

Filings **≤ 2022** → train, **2023+** → test.  
This prevents look-ahead bias — the model is evaluated only on filings filed after the training period.


In [ ]:
import os

TRAIN_CUTOFF = 2022   # filings up to and including 2022 → train; 2023+ → test

train_df = long_df[long_df["year"] <= TRAIN_CUTOFF].copy().reset_index(drop=True)
test_df  = long_df[long_df["year"] >  TRAIN_CUTOFF].copy().reset_index(drop=True)

print(f"Train (≤ {TRAIN_CUTOFF}) : {len(train_df):>8,} sentences")
print(f"Test  ({TRAIN_CUTOFF+1}+)  : {len(test_df):>8,} sentences")
print()
print("Train label distribution:")
print(train_df["label"].value_counts().to_string())
print()
print("Test label distribution:")
print(test_df["label"].value_counts().to_string())

# ── Build HuggingFace Datasets ────────────────────────────────────────────────
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

# num_proc parallelises tokenisation across CPU cores
N_PROC = max(1, os.cpu_count() - 1)

hf_train = HFDataset.from_dict(
    {"text": train_df["text"].tolist(), "labels": train_df["label_id"].tolist()}
)
hf_test = HFDataset.from_dict(
    {"text": test_df["text"].tolist(), "labels": test_df["label_id"].tolist()}
)

hf_train = hf_train.map(tokenize, batched=True, num_proc=N_PROC, remove_columns=["text"])
hf_test  = hf_test.map(tokenize,  batched=True, num_proc=N_PROC, remove_columns=["text"])

# Cache tokenised datasets to disk so re-runs skip retokenisation
hf_train.save_to_disk("./hf_cache/train")
hf_test.save_to_disk("./hf_cache/test")

print(f"\nHF train size : {len(hf_train):,}")
print(f"HF test  size : {len(hf_test):,}")
print(f"Tokenised with {N_PROC} workers; cached to ./hf_cache/")


## 6. Training Hyperparameters

Only **4 training hyperparameters** are set — the minimum needed for a sound fine-tuning run:

| Hyperparameter                | Value  | Why this value                                                                                                                   |
| ----------------------------- | ------ | -------------------------------------------------------------------------------------------------------------------------------- |
| `learning_rate`               | `2e-5` | Standard BERT fine-tuning range (1e-5–5e-5). Too high → catastrophic forgetting of pre-trained weights; too low → barely adapts. |
| `num_train_epochs`            | `3`    | BERT fine-tuning papers consistently find 3 epochs optimal for downstream tasks — more epochs overfit on small datasets.         |
| `per_device_train_batch_size` | `16`   | MPS memory-friendly (adjust to 8 if OOM). Larger batches → more stable gradients but more memory.                                |
| `weight_decay`                | `0.01` | L2 regularisation on all non-bias parameters. Prevents overfitting without needing dropout tuning.                               |


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {"macro_f1": macro_f1}


# ── Mixed-precision flags (device-aware) ──────────────────────────────────────
use_fp16 = DEVICE.type == "cuda"        # fp16 on CUDA (fastest, well-supported)
use_bf16 = DEVICE.type == "cpu"         # bf16 on CPU (avoids fp16 instability)
# MPS: both flags False — Apple's MPS kernels are still maturing for mixed precision

# ── Gradient accumulation lets us use a larger effective batch without extra RAM
# Effective batch = per_device_train_batch_size × gradient_accumulation_steps
GRAD_ACCUM = 4      # effective batch = 16 × 4 = 64

# ── Enable gradient checkpointing to trade compute for memory ─────────────────
model.gradient_checkpointing_enable()

# ── 4 training hyperparameters ────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./finbert_mda_checkpoints",
    # 1. Learning rate + warmup
    learning_rate=2e-5,
    warmup_ratio=0.06,              # linear warmup for first 6 % of steps
    # 2. Epochs
    num_train_epochs=3,
    # 3. Batch size + gradient accumulation
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,  # no gradients stored → larger batch is fine
    gradient_accumulation_steps=GRAD_ACCUM,
    # 4. Regularisation
    weight_decay=0.01,
    # ── Speed / memory ────────────────────────────────────────────────────────
    fp16=use_fp16,
    bf16=use_bf16,
    dataloader_num_workers=max(1, os.cpu_count() // 2),  # parallel data loading
    dataloader_pin_memory=(DEVICE.type == "cuda"),       # pin pages only for CUDA
    eval_accumulation_steps=16,     # stream eval predictions to CPU in chunks
    # ── Bookkeeping ───────────────────────────────────────────────────────────
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,             # keep only the 2 best checkpoints on disk
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=100,
    seed=SEED,
    use_mps_device=(DEVICE.type == "mps"),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_test,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

n_eff = training_args.per_device_train_batch_size * GRAD_ACCUM
print(f"Effective batch size : {n_eff}  ({training_args.per_device_train_batch_size} × {GRAD_ACCUM} accum)")
print(f"Mixed precision      : fp16={use_fp16}  bf16={use_bf16}")
print(f"Dataloader workers   : {training_args.dataloader_num_workers}")
print("Trainer ready. Starting fine-tuning...")
trainer.train()


## 7. Evaluation on Test Set


In [ ]:
preds_out = trainer.predict(hf_test)
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = preds_out.label_ids

pred_labels = [ID2LABEL[p] for p in y_pred]
true_labels = [ID2LABEL[t] for t in y_true]

print("=== Fine-tuned FinBERT on MD&A — Test Set (2023+) ===")
print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=["positive", "neutral", "negative"],
        zero_division=0,
    )
)

macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Macro F1: {macro_f1:.4f}")
